In [ ]:
import rasterio 
import numpy as np
from sklearn.cluster import DBSCAN
import geopandas as gpd
from shapely.geometry import Point


# ocitavanje rastera i dohvatanje podataka
raster = rasterio.open(r"C:\Users\Svetozar Perisin\GIS Projekte\fruskac_final.tif")
data = raster.read(1)

# filtriranje no data podataka i stavljanje thresholda
mask = (data != raster.nodata) & (data < -30)
rows, cols = np.where(mask)
values = data[rows, cols]
weights = np.abs(values)

# pretvaranje u geokordinate
xs, ys = rasterio.transform.xy(raster.transform, rows, cols)
coords = np.column_stack((xs, ys))

# primena DBSCAN alogritma
dbscan = DBSCAN(eps=300, min_samples=35).fit(coords, sample_weight=weights)
labels = dbscan.labels_


#geodata frame pravljenje 
points = [Point(x, y) for x, y in coords]

gdf = gpd.GeoDataFrame(
    {"cluster": labels},
    geometry=points,
    crs=raster.crs
)

#ukloni noise 
gdf = gdf[gdf["cluster"] >= 0]

#napravi zone 
zones = gdf.dissolve(by="cluster")
zones["geometry"] = zones.convex_hull

#export 
zones.to_file("dbscan_hotspots_eps300_min35.geojson", driver="GeoJSON")

print("✅ Gotovo! GeoJSON napravljen.")
